In [ ]:
!pip install numpy pandas matplotlib scipy yfinance arch statsmodels -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Chart style
BLUE = '#1A3A6E'; RED = '#DC3545'; GREEN = '#2E7D32'
ORANGE = '#E67E22'; GRAY = '#666666'; PURPLE = '#8E44AD'

plt.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none',
    'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.grid': False, 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'legend.fontsize': 9, 'xtick.labelsize': 9,
    'ytick.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.0, 'axes.linewidth': 0.6,
    'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.edgecolor': 'none',
    'figure.dpi': 150,
})

def save_chart(fig, name):
    fig.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True, dpi=150)
    fig.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved: {name}')

def legend_bottom(ax, ncol=2, y=-0.18):
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)

In [ ]:
import yfinance as yf
from arch import arch_model

# Download S&P 500 data
sp = yf.download('^GSPC', start='2000-01-01', end='2025-12-31', progress=False)
if isinstance(sp.columns, pd.MultiIndex):
    sp.columns = sp.columns.get_level_values(0)
returns = (sp['Close'].pct_change() * 100).dropna()
returns = pd.Series(returns.values, index=returns.index, name='returns')
print(f"S&P 500: {len(returns)} observations")

In [ ]:
# Fit GARCH(1,1)
model = arch_model(returns, vol='Garch', p=1, q=1, dist='t')
res = model.fit(disp='off')
print(res.summary().tables[1])

# Conditional volatility
cond_vol = res.conditional_volatility

# Unconditional volatility
omega = res.params['omega']
alpha = res.params['alpha[1]']
beta = res.params['beta[1]']
uncond_var = omega / (1 - alpha - beta)
uncond_vol = np.sqrt(uncond_var)
print(f"\nUnconditional volatility: {uncond_vol:.4f}%")
print(f"Persistence (α+β): {alpha+beta:.4f}")

In [ ]:
# Plot conditional vs unconditional volatility
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cond_vol.index, cond_vol.values, color=BLUE, lw=0.6, alpha=0.8,
        label=f'Conditional σ_t (GARCH)')
ax.axhline(uncond_vol, color=RED, lw=1.5, ls='--',
           label=f'Unconditional σ = {uncond_vol:.2f}%')
ax.axhline(returns.std(), color=GREEN, lw=1.5, ls=':',
           label=f'Sample Std Dev = {returns.std():.2f}%')

ax.set_ylabel('Volatility (%)')
ax.set_title('Conditional vs Unconditional Volatility — S&P 500', fontweight='bold')
legend_bottom(ax, ncol=3, y=-0.15)
save_chart(fig, 'garch_cond_vs_uncond')
plt.show()